# 03a Decompression relay with PVK thermal reference

Notebook for the decompression relay function. Last updated 10/9/2024.

Start with dependencies:

In [1]:
import numpy as np
import pyDEW
import os
import matplotlib.pyplot as plt
from thermoengine import model
from copy import deepcopy
import pandas as pd
import linecache
from scipy.interpolate import LinearNDInterpolator
from timeit import default_timer as timer
import warnings

# Additional functions for pyDEW
import pyDEW_functions as pDF

# Structural functions for the relay
import relay_functions as rF

# Folder to export results
if not os.path.exists("./data"):
    os.makedirs("./data")

# pyDEW flags up a few things when the system is created - this counteracts it
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

Create our pyDEW system (required for DATA0 and DATA1 files).

In [2]:
# Adjust here the basis species in the DEW system - deepcopy to avoid mucking up defaults
defaultBasisSpecies = pyDEW.defaultsystem.basis_species_names
basisSpecies = deepcopy(defaultBasisSpecies)

defaultOtherSpecies = pyDEW.defaultsystem.other_species_names
otherSpecies = deepcopy(defaultOtherSpecies)

# Add missing Mg complex to the other species
otherSpecies.append('MG(OH)2(AQ)')

# Switch basis species - replace CO2(AQ) with CO3-2 directly to avoid pyDEW issues
basisSpecies[basisSpecies.index('CO2(AQ)')] = 'CO3-2'

# Create DEW system
dew_system = pyDEW.System(basis_species=basisSpecies, other_species=otherSpecies)


Create the reactants that will fill our column.

In [3]:
# List of reactants formed from mineral dictionaries
olvDict = {'reactant': 'OLIVINE(SS)',
           'moles': 5.9,
           'composition': {'FORSTERITE': 0.9, 'FAYALITE': 0.1}}

opxDict = {'reactant': 'ORTHOPYROXENE(SS)',
           'moles': 1.9,
           'composition': {'ENSTATITE_OR': 0.9, 'FERROSILITE': 0.1}}

cpxDict = {'reactant': 'CLINOPYROXENE(SS)',
           'moles': 0.2,
           'composition': {'DIOPSIDE': 0.9, 'HEDENBERGITE': 0.1}}

spnDict = {'reactant': 'SPINEL',
           'moles': 0.04}

# mgtDict = {'reactant': 'MAGNETITE',
#            'moles': 0.04}

startingReactant = [olvDict, opxDict, cpxDict, spnDict] #, mgtDict]


Create our uniform input from a given string

In [4]:
searchString = 'Marianas'
VKColumnFiles = os.listdir('VK_columns/')
VKColumnFiles = [i for i in VKColumnFiles if '.csv' in i and '0.05' not in i]
VKColumnFiles.sort()
VKColumnFilesLabels = [i.replace('.csv','') for i in VKColumnFiles]
fluidRef = [1.0, 1.0, 1.0, 1.0, 2.620431408, 1.432721913, 1.738881725, 4.799669867]
fluidRef = [int(1000*i) for i in fluidRef]
print(VKColumnFiles)
print(VKColumnFilesLabels)
print(fluidRef)

['04_Cascadia_0.csv', '04_Cascadia_1.csv', '04_Cascadia_2.csv', '04_Cascadia_3.csv', '04_Cascadia_3x.csv', '04_Cascadia_4.csv', '04_Cascadia_5.csv', '04_Cascadia_6.csv', '04_Cascadia_7.csv', '41_N_Marianas_4.csv', '41_N_Marianas_5.csv', '41_N_Marianas_6.csv', '41_N_Marianas_7.csv']
['04_Cascadia_0', '04_Cascadia_1', '04_Cascadia_2', '04_Cascadia_3', '04_Cascadia_3x', '04_Cascadia_4', '04_Cascadia_5', '04_Cascadia_6', '04_Cascadia_7', '41_N_Marianas_4', '41_N_Marianas_5', '41_N_Marianas_6', '41_N_Marianas_7']
[1000, 1000, 1000, 1000, 2620, 1432, 1738, 4799]


In [5]:
for i in range(len(VKColumnFiles)):
    columnDF = pd.read_csv(f'VK_columns/{VKColumnFiles[i]}')
    pressures = columnDF['P (kbar)'].tolist()
    pressures = [i*1000 for i in pressures]  # convert to bars
    temperatures = columnDF['T (K)'].tolist()
    fluidComposition = rF.createHybridFluid([1,1,1], pressures[0], temperatures[0])
    startingComposition = rF.createUniformInput(startingReactant, dew_system,
                                             1000.0, pressures, temperatures, startingTime=0)
    rF.decompressionRelay(startingComposition, fluidComposition[0], fluidComposition[1],
                          dew_system, timesteps=10, fluidMass=fluidRef[i], terminalFRR=0.2,
                          exportPath=f'data/{VKColumnFilesLabels[i]}_mix_VAR_t0.2')

-----RELAY STARTING-----
Relay function initiated
Starting from scratch...


/opt/conda/lib/python3.9/site-packages/pyDEW/eq6_output.py:46: UserWarning: The pH output could not be read.
  warn("The pH output could not be read.")


No mineral product found.


/opt/conda/lib/python3.9/site-packages/pyDEW/eq6_output.py:46: UserWarning: The pH output could not be read.
  warn("The pH output could not be read.")


No mineral product found.


/opt/conda/lib/python3.9/site-packages/pyDEW/eq6_output.py:46: UserWarning: The pH output could not be read.
  warn("The pH output could not be read.")


... (864 outputs left out)

No mineral product found.


/opt/conda/lib/python3.9/site-packages/pyDEW/eq6_output.py:46: UserWarning: The pH output could not be read.
  warn("The pH output could not be read.")


No mineral product found.
t=4, calculations=261, errors=41 of which FRR passes=0
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
No mineral product found.
t=5, calculations=268, errors=34 of which FRR passes=0
No mineral product found.
No mineral product found.
No mineral product found.
No min

IndexError: list index out of range

In [ ]:
# for i in range(len(VKColumnFiles)):
#     columnDF = pd.read_csv(f'VK_columns/{VKColumnFiles[i]}')
#     pressures = columnDF['P (kbar)'].tolist()
#     pressures = [i*1000 for i in pressures]  # convert to bars
#     temperatures = columnDF['T (K)'].tolist()
#     fluidComposition = rF.createHybridFluid([0,0,1], pressures[0], temperatures[0])
#     startingComposition = rF.createUniformInput(startingReactant, dew_system,
#                                              1000.0, pressures, temperatures, startingTime=0)
#     rF.decompressionRelay(startingComposition, fluidComposition[0], fluidComposition[1],
#                           dew_system, timesteps=10, fluidMass=fluidRef[i], terminalFRR=0.2,
#                           exportPath=f'data/{VKColumnFilesLabels[i]}_sed_VAR')

In [ ]:
# columnDF = pd.read_csv(f'VK_columns/04_Cascadia_3x.csv')
# pressures = columnDF['P (kbar)'].tolist()
# pressures = [i*1000 for i in pressures]  # convert to bars
# temperatures = columnDF['T (K)'].tolist()
# fluidComposition = rF.createHybridFluid([1,1,1], pressures[0], temperatures[0])
# startingComposition = rF.createUniformInput(startingReactant, dew_system,
#                                          1000.0, pressures, temperatures, startingTime=0)
# rF.decompressionRelay(startingComposition, fluidComposition[0], fluidComposition[1],
#                       dew_system, timesteps=10, fluidMass=287.212, terminalFRR=0.2,
#                       exportPath=f'data/04_Cascadia_3x_mix_VAR_02')